<a href="https://colab.research.google.com/github/leilacielok/market_basket_analysis/blob/main/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algorithms for Massive Data - Market-Basket Analysis

In [1]:
import os
import pandas as pd

from google.colab import userdata
from itertools import combinations
from collections import Counter

In [2]:
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")


# SOSTITUIRE CON: Insert your Kaggle credentials before running
# os.environ['KAGGLE_USERNAME'] = "xxxx"
# os.environ['KAGGLE_KEY'] = "xxxx"

!kaggle datasets download -d harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows
!unzip -o -q imdb-dataset-of-top-1000-movies-and-tv-shows.zip -d imdb_data

print("Dataset downloaded and extracted.")

Dataset URL: https://www.kaggle.com/datasets/harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows
License(s): CC0-1.0
100% 175k/175k [00:00<00:00, 24.2MB/s]

Dataset downloaded and extracted.


In [3]:
csv_path = os.path.join(
    "imdb_data",
    "imdb_top_1000.csv"
)

df = pd.read_csv(csv_path)

print(df.shape)
df.head()

(1000, 16)


,Poster_Link,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Overview,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross
0,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994,A,142 min,Drama,9.3,Two imprisoned men bond over a number of years...,80.0,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler,2343110,"28,341,469"
1,https://m.media-amazon.com/images/M/MV5BM2MyNj...,The Godfather,1972,A,175 min,"Crime, Drama",9.2,An organized crime dynasty's aging patriarch t...,100.0,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,"134,966,411"
2,https://m.media-amazon.com/images/M/MV5BMTMxNT...,The Dark Knight,2008,UA,152 min,"Action, Crime, Drama",9.0,When the menace known as the Joker wreaks havo...,84.0,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,"534,858,444"
3,https://m.media-amazon.com/images/M/MV5BMWMwMG...,The Godfather: Part II,1974,A,202 min,"Crime, Drama",9.0,The early life and career of Vito Corleone in ...,90.0,Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton,1129952,"57,300,000"
4,https://m.media-amazon.com/images/M/MV5BMWU4N2...,12 Angry Men,1957,U,96 min,"Crime, Drama",9.0,A jury holdout attempts to prevent a miscarria...,96.0,Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler,689845,"4,360,000"


In [4]:
# Minimum support threshold
MIN_SUPPORT_RATIO = 0.01
MIN_SUPPORT_COUNT = int(len(df) * MIN_SUPPORT_RATIO)

print("Minimum support count:", MIN_SUPPORT_COUNT)

Minimum support count: 10


In [6]:
# Basket construction
STAR_COLS = ["Star1", "Star2", "Star3", "Star4"]

baskets = []

for _, row in df.iterrows():
    basket = []

    for col in STAR_COLS:
        actor = row[col]
        basket.append(actor)

    baskets.append(set(basket))

print("Number of baskets:", len(baskets))
print("Example basket:", baskets[0])

Number of baskets: 1000
Example basket: {'Morgan Freeman', 'Bob Gunton', 'Tim Robbins', 'William Sadler'}


In [7]:
frequent_itemsets = {}

# 1-itemsets
item_counts = Counter()

for basket in baskets:
    for item in basket:
        item_counts[frozenset([item])] += 1

current_frequent = {
    itemset: count
    for itemset, count in item_counts.items()
    if count >= MIN_SUPPORT_COUNT
}

frequent_itemsets[1] = current_frequent

print("Frequent 1-itemsets:", len(current_frequent))


# k-itemsets
k = 2

while current_frequent:
    candidate_counts = Counter()

    frequent_items_previous = list(current_frequent.keys())

    candidates = set()

    for itemset1 in frequent_items_previous:
        for itemset2 in frequent_items_previous:
            candidate = itemset1.union(itemset2)

            if len(candidate) == k:
                candidates.add(candidate)

    for basket in baskets:
        for candidate in candidates:
            if candidate.issubset(basket):
                candidate_counts[candidate] += 1

    current_frequent = {
        itemset: count
        for itemset, count in candidate_counts.items()
        if count >= MIN_SUPPORT_COUNT
    }

    if current_frequent:
        frequent_itemsets[k] = current_frequent
        print(f"Frequent {k}-itemsets:", len(current_frequent))

    k += 1

Frequent 1-itemsets: 9


In [8]:
# Display frequent itemsets

for size, itemsets in frequent_itemsets.items():
    print("\n" + "=" * 50)
    print(f"Frequent itemsets of size {size}")
    print("=" * 50)

    sorted_itemsets = sorted(
        itemsets.items(),
        key=lambda x: x[1],
        reverse=True
    )

    for itemset, count in sorted_itemsets[:20]:
        print(set(itemset), "-> support count:", count)


Frequent itemsets of size 1
{'Robert De Niro'} -> support count: 17
{'Tom Hanks'} -> support count: 14
{'Al Pacino'} -> support count: 13
{'Brad Pitt'} -> support count: 12
{'Clint Eastwood'} -> support count: 12
{'Christian Bale'} -> support count: 11
{'Leonardo DiCaprio'} -> support count: 11
{'Matt Damon'} -> support count: 11
{'James Stewart'} -> support count: 10
